# KTPM-BTL — Pha 1: SQLite baseline trên Kaggle CPU

Chọn **Accelerator: None (CPU)** và **Internet: On**. Dùng **Run All** rồi **Save Version → Save & Run All** để lưu kết quả cùng notebook. Không dùng số liệu local làm bằng chứng Kaggle.

Notebook lấy mã nguồn công khai, build/kiểm thử JDK 17 và chạy Locust 50 users trong 120 giây, warm-up 15 giây. Backend và load generator cùng dùng CPU của notebook. Database mới chỉ chứa dữ liệu giả lập. Các mức tải và ngưỡng lỗi là chỉ tiêu của nhóm, không phải ngưỡng do đề bài quy định.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, datetime, shutil
assert Path('/kaggle/working').is_dir(), 'Notebook này cần chạy trên Kaggle'
assert sys.version_info >= (3, 11), 'Cần Python 3.11+ cho dependency đã pin'
REPOSITORY = 'https://github.com/FlowBoat123/KTPM-BTL.git'
SOURCE_REF = 'codex/phase1-sqlite-kaggle'
ROOT = Path('/kaggle/working/KTPM-BTL')
if not (ROOT / '.git').exists():
    subprocess.run(['git','clone','--branch',SOURCE_REF,'--single-branch',REPOSITORY,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',SOURCE_REF],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('Code được đo:', COMMIT)
print('GPU/TPU:', os.environ.get('CUDA_VISIBLE_DEVICES','none'))
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
assert gpu is None or gpu.returncode != 0 or not gpu.stdout.strip(), 'Hãy chọn Accelerator None (CPU)' 

In [ ]:
# Cố định JDK 17. Lưu build log và bản dependency để có thể đối chiếu.
JAVA_HOME = Path('/usr/lib/jvm/java-17-openjdk-amd64')
if not (JAVA_HOME / 'bin/java').exists():
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','openjdk-17-jdk-headless'],check=True)
os.environ['JAVA_HOME'] = str(JAVA_HOME)
os.environ['PATH'] = str(JAVA_HOME / 'bin') + os.pathsep + os.environ['PATH']
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-load.txt')],check=True)
(ROOT/'mvnw').chmod(0o755)
BUILD_LOG = Path('/kaggle/working/build.log')
with BUILD_LOG.open('w') as log:
    build = subprocess.run([str(ROOT/'mvnw'),'-B','verify'],cwd=ROOT,stdout=log,stderr=subprocess.STDOUT)
print('\n'.join(BUILD_LOG.read_text().splitlines()[-25:]))
assert build.returncode == 0, 'Build/test thất bại: xem build.log'

In [ ]:
USERS = 50
SPAWN_RATE = 5
DURATION_SECONDS = 120
WARMUP_SECONDS = 15
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/kaggle/working') / ('sqlite-baseline-' + RUN_ID)
command = [sys.executable,str(ROOT/'scripts/run_load_test.py'),'--environment','kaggle','--users',str(USERS),'--spawn-rate',str(SPAWN_RATE),'--duration',str(DURATION_SECONDS),'--warmup',str(WARMUP_SECONDS),'--output',str(OUTPUT)]
result = subprocess.run(command,cwd=ROOT)
assert (OUTPUT/'metadata.json').exists(), 'Không có metadata kết quả'
print(json.dumps(json.loads((OUTPUT/'metadata.json').read_text()),indent=2,ensure_ascii=False))
assert result.returncode == 0, 'Chạy chưa đạt chỉ tiêu hoặc gặp lỗi; giữ nguyên log/CSV để phân tích'

In [ ]:
from IPython.display import display, Markdown, Image, FileLink
display(Markdown((OUTPUT/'REPORT.md').read_text()))
display(Image(filename=str(OUTPUT/'p95_latency.png')))
display(FileLink(str(OUTPUT.with_suffix('.zip'))))
display(FileLink(str(OUTPUT/'locust_stats.csv')))
display(FileLink(str(OUTPUT/'metadata.json')))
print('Nhớ Save Version → Save & Run All để giữ notebook và output làm bằng chứng.')

## Cách đọc kết quả

- `preflight.json`: xác minh đặt lịch, phân công, hoàn thành, quyền truy cập, chống trùng giờ và thu hồi JWT trước phép đo.
- `metadata.json`: commit, môi trường, CPU/RAM, cgroup quota, Java/Python/dependency, SQLite JDBC, pool, JVM heap và thời gian chạy.
- `locust_stats.csv`, `locust_stats_history.csv`: request count, failures, RPS và phân vị độ trễ.
- `summary.json`, `REPORT.md`, `report.html`, `p95_latency.png`: tổng hợp và đồ thị.
- `.zip`: gói bằng chứng; không chứa file database hay token.

`target_met` chỉ kiểm tra ngưỡng lỗi nhóm chọn (mặc định ≤1%). Báo cáo vẫn phải nêu cả RPS/độ trễ và giới hạn phép đo. Load generator và backend chia sẻ máy; trọng số chọn user/task không tương đương tỷ lệ HTTP request. Kịch bản tải không đo phân công đồng thời. Giữ nguyên phần cứng, workload, heap và dữ liệu khởi tạo khi so sánh Pha 2.